# Proyecto Final de Minería de Datos, Fase de Despliegue con Interfaz Gráfica Streamlit (CRISP-DM)
## Dataset: Predicción de Abandono de Clientes de Tarjetas de Crédito (*Credit Card Churn*)

Este notebook documenta y ejecuta la **Fase 6 (Despliegue)** de la metodología **CRISP-DM**:
1. **Inspección y Verificación del `Pipeline` Empaquetado (`modelo_churn_pipeline.joblib`)**
2. **Simulación de Inferencia sobre Nuevos Perfiles de Clientes (Datos Futuros)**
3. **Arquitectura de la Aplicación Interactiva en Streamlit (`app_streamlit.py`)**
4. **Plan de Monitoreo de Deriva de Datos (*Data Drift* / *Concept Drift*) y Cronograma de Mantenimiento**


In [1]:
import joblib
import pandas as pd
import numpy as np

bundle = joblib.load("modelo_churn_pipeline.joblib")
pipeline = bundle["pipeline"]

print("=== PIPELINE CARGADO EXITOSAMENTE PARA PRODUCCIÓN ===")
print(f"Pasos del Pipeline: {[step[0] for step in pipeline.steps]}")
print(f"Método de optimización ganador: {bundle['optimization_method']}")
print(f"Hiperparámetros óptimos del clasificador XGBoost: {bundle['best_params']}")
print(f"Métricas validadas en Test (30%): {bundle['test_metrics']}")


=== PIPELINE CARGADO EXITOSAMENTE PARA PRODUCCIÓN ===
Pasos del Pipeline: ['unknown_to_nan', 'imputer', 'features', 'preprocessor', 'smote', 'classifier']
Método de optimización ganador: Optimización Bayesiana (Optuna TPE)
Hiperparámetros óptimos del clasificador XGBoost: {'n_estimators': 300, 'max_depth': 5, 'learning_rate': 0.1266, 'subsample': 0.9278, 'colsample_bytree': 0.7052}
Métricas validadas en Test (30%): {'Accuracy': 0.9657, 'Precision': 0.9, 'Recall': 0.8852, 'F1-Score': 0.8926, 'ROC-AUC': 0.9912}


In [2]:
nuevos_clientes = pd.DataFrame([
    {
        "Perfil": "Cliente A (Activo y Vinculado)",
        "Customer_Age": 45, "Gender": "M", "Dependent_count": 2,
        "Education_Level": "Graduate", "Marital_Status": "Married",
        "Income_Category": "$80K - $120K", "Card_Category": "Blue",
        "Months_on_book": 36, "Total_Relationship_Count": 5,
        "Months_Inactive_12_mon": 1, "Contacts_Count_12_mon": 1,
        "Credit_Limit": 12000.0, "Total_Revolving_Bal": 1450,
        "Total_Amt_Chng_Q4_Q1": 0.85, "Total_Trans_Amt": 4800,
        "Total_Trans_Ct": 78, "Total_Ct_Chng_Q4_Q1": 0.82,
        "Avg_Utilization_Ratio": 0.121
    },
    {
        "Perfil": "Cliente B (Baja Actividad y Múltiples Reclamos)",
        "Customer_Age": 48, "Gender": "F", "Dependent_count": 3,
        "Education_Level": "High School", "Marital_Status": "Single",
        "Income_Category": "Less than $40K", "Card_Category": "Blue",
        "Months_on_book": 39, "Total_Relationship_Count": 2,
        "Months_Inactive_12_mon": 4, "Contacts_Count_12_mon": 4,
        "Credit_Limit": 3200.0, "Total_Revolving_Bal": 0,
        "Total_Amt_Chng_Q4_Q1": 0.42, "Total_Trans_Amt": 1650,
        "Total_Trans_Ct": 32, "Total_Ct_Chng_Q4_Q1": 0.38,
        "Avg_Utilization_Ratio": 0.000
    }
])

X_nuevos = nuevos_clientes.drop(columns=["Perfil"])
preds = pipeline.predict(X_nuevos)
probs = pipeline.predict_proba(X_nuevos)[:, 1]

res_inferencia = nuevos_clientes[["Perfil", "Total_Trans_Ct", "Total_Revolving_Bal", "Months_Inactive_12_mon", "Contacts_Count_12_mon"]].copy()
res_inferencia["Predicción"] = np.where(preds == 1, "Attrited Customer (Abandona)", "Existing Customer (Retenido)")
res_inferencia["Probabilidad_Churn (%)"] = np.round(probs * 100, 2)
print("=== PRUEBA DE INFERENCIA EN DATOS FUTUROS ===")
print(res_inferencia.to_string(index=False))


=== PRUEBA DE INFERENCIA EN DATOS FUTUROS ===
                                         Perfil  Total_Trans_Ct  Total_Revolving_Bal  Months_Inactive_12_mon  Contacts_Count_12_mon                   Predicción  Probabilidad_Churn (%)
                 Cliente A (Activo y Vinculado)              78                 1450                       1                      1 Existing Customer (Retenido)                     0.0
Cliente B (Baja Actividad y Múltiples Reclamos)              32                    0                       4                      4 Attrited Customer (Abandona)                   100.0


In [3]:
# Prueba con datos crudos del CSV original, que traen Unknown en tres columnas
df_original = pd.read_csv("credit_card_churn.csv")
columnas_unknown = ['Education_Level', 'Marital_Status', 'Income_Category']
filas_unknown = pd.concat([df_original[df_original[c] == 'Unknown'].head(2) for c in columnas_unknown]).drop_duplicates().head(5)
X_unknown = filas_unknown[bundle["raw_columns"]]
assert all((X_unknown[c] == 'Unknown').any() for c in columnas_unknown)

res_unknown = filas_unknown[columnas_unknown + ['Attrition_Flag']].copy()
res_unknown["Predicción"] = np.where(pipeline.predict(X_unknown) == 1, "Attrited Customer", "Existing Customer")
res_unknown["Probabilidad_Churn (%)"] = np.round(pipeline.predict_proba(X_unknown)[:, 1] * 100, 2)
print("=== PRUEBA CON FILAS CRUDAS DEL CSV ORIGINAL CON Unknown ===")
print(res_unknown.to_string())

# El archivo completo tampoco debe fallar ni producir probabilidades inválidas
X_completo = df_original[bundle["raw_columns"]]
probs_completo = pipeline.predict_proba(X_completo)[:, 1]
assert np.isfinite(probs_completo).all() and ((probs_completo >= 0) & (probs_completo <= 1)).all()
print(f"\nArchivo completo: {len(X_completo)} filas, {int((X_completo == 'Unknown').any(axis=1).sum())} con Unknown, todas calificadas")

=== PRUEBA CON FILAS CRUDAS DEL CSV ORIGINAL CON Unknown ===
   Education_Level Marital_Status Income_Category     Attrition_Flag         Predicción  Probabilidad_Churn (%)
6          Unknown        Married         $120K +  Existing Customer  Existing Customer                    0.05
11         Unknown        Married     $40K - $60K  Existing Customer  Existing Customer                    0.00
3      High School        Unknown  Less than $40K  Existing Customer  Existing Customer                    0.02
7      High School        Unknown     $60K - $80K  Existing Customer  Existing Customer                    0.00
19        Graduate        Married         Unknown  Existing Customer  Existing Customer                    0.01

Archivo completo: 10127 filas, 3046 con Unknown, todas calificadas


---
## 3. Ejecución de la Interfaz Gráfica Web en Streamlit (`app_streamlit.py`)

La aplicación de despliegue se encuentra implementada en el archivo `app_streamlit.py` e incluye:
- **Pestaña 1, Predicción Individual en Tiempo Real**: Formulario interactivo dividido en *Datos Demográficos*, *Relación con el Banco* y *Comportamiento Transaccional*, variables de ingeniería calculadas por el propio `Pipeline`, probabilidad estimada de fuga y recomendación comercial personalizada.
- **Pestaña 2, Predicción Masiva por Lote (CSV)**: Carga de archivos `.csv` de clientes para calificación masiva y descarga inmediata de resultados.

Para iniciar la aplicación interactiva localmente desde la terminal:
```bash
streamlit run app_streamlit.py
```


---
## 4. Plan de Monitoreo de Deriva de Datos (*Data Drift* / *Concept Drift*) y Cronograma de Mantenimiento

### Valores de referencia
El desempeño de referencia es el del `Pipeline` en el 30% de prueba, con `F1-Score` de 0.8926, `Recall` de 0.8852, `Precision` de 0.9000 y `ROC-AUC` de 0.9912. En ese mismo conjunto el modelo alerta sobre el 15.81% de los clientes, cuando la tasa real de abandono es 16.07%. Los umbrales de este plan son una propuesta inicial, porque el proyecto no cuenta con datos de producción, y deben ajustarse cuando exista historial real.

### Deriva de datos
Cada mes se compara la distribución de las 18 predictoras crudas con la del conjunto de entrenamiento mediante el índice de estabilidad poblacional (PSI). Según la convención habitual, un PSI menor a 0.10 se considera estable, entre 0.10 y 0.25 amerita revisión y por encima de 0.25 indica un cambio importante. La vigilancia empieza por `Total_Trans_Ct` y `Total_Trans_Amt`, que son las variables de mayor peso en el modelo.

También se vigilan dos situaciones que el `Pipeline` resuelve sin generar errores y que por eso pueden pasar inadvertidas. La primera es el porcentaje de `Unknown`, que en el archivo original es 15.00% en `Education_Level`, 10.98% en `Income_Category` y 7.40% en `Marital_Status`, porque un aumento significa que más clientes se califican con la moda en lugar de su valor real. La segunda es la aparición de categorías nuevas, que el `OrdinalEncoder` convierte en -1 y el `OneHotEncoder` deja en ceros, por lo que cualquier categoría no vista en el entrenamiento debe registrarse y revisarse.

### Deriva de concepto
El abandono solo se confirma tiempo después de la predicción, así que las métricas reales se calculan por cohortes cuando ya se conoce el desenlace de cada cliente. Se propone generar una alerta si el `Recall` baja de 0.85, que es el mínimo exigido en la línea base del informe, o si el `F1-Score` baja de 0.87, que corresponde aproximadamente a su valor de referencia menos dos desviaciones estándar entre folds. Mientras llegan esas etiquetas, la tasa de alertas y la distribución de la probabilidad estimada sirven como señal temprana, porque un cambio fuerte frente al 15.81% de referencia puede anticipar una deriva.

### Cronograma de mantenimiento

| Frecuencia | Actividad |
| :--- | :--- |
| Mensual | Calcular PSI, porcentaje de `Unknown`, categorías nuevas, tasa de alertas y distribución de probabilidades |
| Trimestral | Calcular `Recall`, `Precision`, `F1-Score` y `ROC-AUC` sobre las cohortes con desenlace conocido |
| Anual | Reentrenar y comparar el modelo nuevo con el vigente sobre un mismo periodo reciente |
| Por evento | Reentrenar antes de lo previsto si se supera un umbral de desempeño, si el PSI de una variable clave pasa de 0.25 o si cambian los productos o las políticas del banco |

### Procedimiento de reentrenamiento
El reentrenamiento repite los notebooks 01 y 02 con datos actualizados e instala las versiones exactas de `requirements.txt`, porque el archivo `modelo_churn_pipeline.joblib` no debe cargarse con otras versiones de las librerías. El modelo nuevo solo reemplaza al vigente si mejora su desempeño en un conjunto de prueba reciente. El archivo `churn_features.py` debe desplegarse siempre junto al modelo, ya que el `Pipeline` guardado depende de sus funciones, y la versión anterior del modelo se conserva para poder revertir el cambio si aparece algún problema.